# WTI Oil Price Forecasting — One Agent, Three Tasks

> **Part 3 of 7.** This notebook builds on the agentic predictor introduced in
> [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb).

A single Analyst Agent — backed by bounded Google Search — answers three tasks
using **one system prompt** and **task-specific user payloads**:

| Stream | Task | Output |
|--------|------|--------|
| A | Trajectory | 5/10/21-day price forecasts |
| B | Binary shock | P(WTI +$5 in 5 days) |
| C | Scenario analysis | Top 3 expert scenarios for 60 days |


In [1]:
# ── Environment setup: load .env, corporate CA, build-cli token refresh ──────
import os
import shutil
import subprocess
import sys
from pathlib import Path

from dotenv import load_dotenv


def _find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists() and (cand / "aieng-forecasting").is_dir():
            return cand
    return Path.cwd().resolve().parents[1]


ROOT = _find_repo_root()
load_dotenv(ROOT / ".env")

# curl_cffi (used by yfinance) bundles its own libcurl and ignores
# pip-system-certs. On macOS, point it at a bundle that includes the corporate CA.
if sys.platform == "darwin" and not os.environ.get("CURL_CA_BUNDLE"):
    import certifi as _certifi

    _combined = Path.home() / ".cache" / "agentic-forecasting" / "combined-ca.pem"
    _combined.parent.mkdir(parents=True, exist_ok=True)
    _kc = subprocess.run(
        ["security", "find-certificate", "-a", "-p", "/Library/Keychains/System.keychain"],
        capture_output=True, text=True,
    )
    _combined.write_text(Path(_certifi.where()).read_text() + "\n" + _kc.stdout)
    os.environ["CURL_CA_BUNDLE"] = str(_combined)
    os.environ["SSL_CERT_FILE"] = str(_combined)
    os.environ["REQUESTS_CA_BUNDLE"] = str(_combined)

# build-cli tokens are short-lived; refresh so GUI-launched kernels (where
# ~/.local/bin is not on PATH) don't silently use a stale key.
_BUILDCLI = next(
    (str(p) for p in [Path.home() / ".local" / "bin" / "build-cli", Path(shutil.which("build-cli") or "")] if p.exists()),
    None,
)
if "build-cli.roche.com" in os.environ.get("PROXY_BASE_URL", ""):
    if _BUILDCLI:
        _result = subprocess.run([_BUILDCLI, "auth", "token"], capture_output=True, text=True, timeout=15)
        if _result.returncode == 0 and _result.stdout.strip():
            os.environ["PROXY_API_KEY"] = _result.stdout.strip()
            print(f"build-cli token refreshed: …{_result.stdout.strip()[-4:]}")
        else:
            print(f"⚠️  build-cli auth token failed: {_result.stderr.strip() or 'no output'}")
            print("    Run 'build-cli login' in a terminal, then restart the kernel.")
    else:
        print("⚠️  build-cli not found — set PROXY_API_KEY manually: export PROXY_API_KEY=$(build-cli auth token)")


build-cli token refreshed: …eS6Q


In [2]:
import json
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display  # noqa: A004


warnings.filterwarnings("ignore")

# ── Model selection ───────────────────────────────────────────────────────────
# Two project models: "anthropic/claude-haiku-4-5-20251001" (lite) and
# "anthropic/claude-sonnet-4-6[1m]" (advanced). Sonnet is the default here; switch to lite
# for faster, cheaper runs.
AGENT_MODEL = "anthropic/claude-sonnet-4-6[1m]"

# ── Cache control ─────────────────────────────────────────────────────────────
# Set to False to force a full end-to-end agent run (ignores all cached results).
USE_CACHE = False

from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_forecasting.analysis import compute_brier_score, trajectory_mae_table
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service, naive_utc_now
from energy_oil_forecasting.paths import (
    PROPHET_SHOCK_TRAJ_CACHE,
    PROPHET_TRAJ_CACHE,
    SCENARIO_CACHE,
    SCENARIO_ORIGIN,
    SHOCK_ANALYST_CACHE,
    SHOCK_HORIZON,
    SHOCK_ORIGINS,
    SHOCK_THRESHOLD,
    TRAJ_AGENT_CACHE,
    TRAJECTORY_ORIGINS,
)
from energy_oil_forecasting.prophet_baseline import (
    check_shock_outcome,
    load_prophet_trajectories,
    prophet_prob_shock,
    wti_series_to_price_df,
)
from energy_oil_forecasting.tasks import TASK_SPECS, build_wti_news_predictor
from energy_oil_forecasting.viz import (
    conf_bar,
    make_shock_comparison_chart,
    make_trajectory_fan_chart,
    prob_bar,
    verdict_label,
)


data_service = build_wti_service()
ctx = data_service.context(as_of=naive_utc_now())
price_df = wti_series_to_price_df(ctx.get_series(WTI_SERIES_ID))

prophet_traj_df = load_prophet_trajectories(price_df, TRAJECTORY_ORIGINS, PROPHET_TRAJ_CACHE)
prophet_shock_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(f"Price history through {price_df.index[-1].date()}")

Loaded 63 Prophet trajectory rows from energy_prophet_trajectories.parquet
Loaded 126 Prophet trajectory rows from energy_shock_prophet_trajectories.parquet
Price history through 2026-07-14


---
## Stream 1 — Trajectory Forecast

Compare Prophet fan charts to the news-grounded agent at three origins.

In [3]:
trajectory_task = ForecastingTask(
    task_id="wti_trajectory_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[5, 10, 21],
    frequency="B",
    description="Trajectory demo for NB3",
)

traj_predictor = build_wti_news_predictor("trajectory", model=AGENT_MODEL)

if USE_CACHE and TRAJ_AGENT_CACHE.exists():
    with open(TRAJ_AGENT_CACHE) as f:
        traj_agent_results = json.load(f)
    print(f"Loaded {len(traj_agent_results)} cached trajectory agent runs.")
else:
    traj_agent_results = []
    for origin in TRAJECTORY_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = traj_predictor.predict(trajectory_task, origin_ctx)
        traj_agent_results.append(
            {
                "origin": str(origin.date()),
                "predictions": [p.model_dump(mode="json") for p in preds],
            }
        )
    with open(TRAJ_AGENT_CACHE, "w") as f:
        json.dump(traj_agent_results, f, indent=2)
    print(f"Saved {len(traj_agent_results)} agent trajectory runs.")

# Summary: agent point forecasts at each origin
print("\nAgent trajectory summary:")
for r in traj_agent_results:
    preds = r["predictions"]
    pts = [f"h{[5, 10, 21][i]}=${preds[i]['payload']['point_forecast']:.1f}" for i in range(len(preds))]
    origin_price_rows = price_df[price_df.index >= pd.Timestamp(r["origin"])]
    origin_price = f"WTI=${origin_price_rows.iloc[0]['price']:.2f}" if not origin_price_rows.empty else ""
    print(f"  {r['origin']}  {origin_price}  {' | '.join(pts)}")


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

Saved 3 agent trajectory runs.

Agent trajectory summary:
  2026-02-02  WTI=$62.14  h5=$64.5 | h10=$63.5 | h21=$62.0
  2026-02-23  WTI=$66.31  h5=$65.5 | h10=$64.5 | h21=$63.0
  2026-03-02  WTI=$71.23  h5=$65.5 | h10=$64.5 | h21=$63.0


In [4]:
# ── I/O inspection: 2026-03-02 — conflict onset, most informative ────────────
INSPECT_ORIGIN = "2026-03-02"
inspect_rec = next((r for r in traj_agent_results if r["origin"] == INSPECT_ORIGIN), None)

if inspect_rec:
    origin_ts = pd.Timestamp(INSPECT_ORIGIN)
    bday_dates = pd.bdate_range(start=origin_ts + pd.offsets.BDay(1), periods=21)
    origin_price_row = price_df[price_df.index >= origin_ts]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")

    preds = inspect_rec["predictions"]
    rationale = preds[0].get("metadata", {}).get("rationale", "") if preds else ""

    table_rows = "| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |\n|---|---|---|---|---|---|\n"
    for i, h in enumerate([5, 10, 21]):
        actual_rows = price_df[price_df.index >= bday_dates[h - 1]]
        actual = float(actual_rows.iloc[0]["price"]) if not actual_rows.empty else float("nan")
        pt = preds[i]["payload"]["point_forecast"]
        q10_val = next(
            (v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.1) < 1e-6), float("nan")
        )
        q90_val = next(
            (v for k, v in preds[i]["payload"]["quantiles"].items() if abs(float(k) - 0.9) < 1e-6), float("nan")
        )
        p_row = prophet_traj_df[(prophet_traj_df["origin"] == origin_ts) & (prophet_traj_df["horizon"] == h)]
        p_yhat = float(p_row.iloc[0]["yhat"]) if not p_row.empty else float("nan")
        table_rows += (
            f"| {h} bdays | **${pt:.1f}** | [{q10_val:.1f} – {q90_val:.1f}] "
            f"| ${actual:.1f} | {pt - actual:+.1f} | {p_yhat - actual:+.1f} |\n"
        )

    display(
        Markdown(
            f"### Stream 1 — I/O Inspection: {INSPECT_ORIGIN}  (WTI ${origin_price:.2f}/bbl)\n\n"
            "Agent and Prophet point forecasts vs realised prices at each horizon.\n\n"
            + table_rows
            + (f"\n> **Agent rationale:** {rationale}" if rationale else "")
        )
    )

### Stream 1 — I/O Inspection: 2026-03-02  (WTI $71.23/bbl)

Agent and Prophet point forecasts vs realised prices at each horizon.

| Horizon | Agent ($) | 80% CI | Actual ($) | Agent err | Prophet err |
|---|---|---|---|---|---|
| 5 bdays | **$65.5** | [60.0 – 72.0] | $94.8 | -29.3 | -30.1 |
| 10 bdays | **$64.5** | [58.5 – 72.5] | $93.5 | -29.0 | -29.1 |
| 21 bdays | **$63.0** | [55.5 – 73.0] | $101.4 | -38.4 | -36.8 |

> **Agent rationale:** Forecast as of 2026-03-01 with last close $65.21 (Feb 26). Key market intelligence: (1) OPEC+ paused output hikes for Q1 2026 after releasing 2.9 mb/d since April 2025 — structural bearish supply overhang persists; (2) IEA Feb 2026 OMR forecasts world output rising 2.4 mb/d in 2026 to 108.6 mb/d; (3) Global inventory builds totaled 477 mb in 2025 with surplus projected at up to 3.84 mb/d in 2026; (4) EIA Feb STEO forecasts Brent averaging $58/bbl in 2026 (WTI ~$55–57); Reuters analyst consensus: WTI avg $60.38/bbl in 2026; (5) Israel-Iran conflict began Feb 28, 2026, adding a $4–10/bbl geopolitical risk premium — no physical supply disruption confirmed as of the as_of date; (6) BNEF baseline Brent $55/bbl in 2026 with tail scenario of $91 if Iran supply fully disrupted; (7) US crude inventories at 442 mb (elevated), Cushing at 25.76 mb; SPR at 396.7 mb. The forecast trajectory reflects a slight near-term geopolitical support near $65–65.50, followed by a gradual reversion toward $63 over one month as the structural bearish supply/demand balance reasserts. Uncertainty increases substantially with horizon, driven by the binary geopolitical outcome (Iran escalation vs. de-escalation) overlaid on the secular bearish fundamental backdrop.

In [5]:
# ── Trajectory fan chart: Prophet fan vs agent error bars at 3 origins ───────
fig = make_trajectory_fan_chart(traj_agent_results, prophet_traj_df, price_df, TRAJECTORY_ORIGINS)
fig.show()

# ── MAE evaluation table ──────────────────────────────────────────────────────
mae_df = trajectory_mae_table(traj_agent_results, prophet_traj_df, price_df)
if not mae_df.empty:
    display(mae_df.drop(columns=["Prophet MAE", "Agent MAE"]))
    mean_mae = mae_df[["Prophet MAE", "Agent MAE"]].mean()
    print(f"\nMean MAE  Prophet: ${mean_mae['Prophet MAE']:.2f}  Agent: ${mean_mae['Agent MAE']:.2f}")

Actual ($) Prophet ($) Agent ($)
Origin     Horizon                                  
2026-02-02 5 bdays        64.4        62.5      64.5
           10 bdays       62.3        63.3      63.5
           21 bdays       74.6        65.2      62.0
2026-02-23 5 bdays        71.2        64.7      65.5
           10 bdays       94.8        64.8      64.5
           21 bdays       92.3        64.5      63.0
2026-03-02 5 bdays        94.8        64.6      65.5
           10 bdays       93.5        64.4      64.5
           21 bdays      101.4        64.6      63.0


Mean MAE  Prophet: $19.19  Agent: $19.54


---
## Stream 2 — Binary Shock Prediction

In [6]:
shock_task = ForecastingTask(
    task_id="wti_upshock_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[SHOCK_HORIZON],
    frequency="B",
    description="Binary upshock demo",
)

shock_predictor = build_wti_news_predictor("shock", model=AGENT_MODEL)

if USE_CACHE and SHOCK_ANALYST_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_results = json.load(f)
    print(f"Loaded {len(shock_results)} cached shock forecasts.")
else:
    shock_results = []
    for origin in SHOCK_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = shock_predictor.predict(shock_task, origin_ctx)
        outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
        shock_results.append(
            {
                "origin": str(origin.date()),
                "probability": preds[0].payload.probability,
                "outcome": outcome,
                "delta": delta,
                "metadata": preds[0].metadata,
            }
        )
    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_results, f, indent=2)

agent_probs = [r["probability"] for r in shock_results]
outcomes = [r["outcome"] for r in shock_results]
print(f"Agent Brier score: {compute_brier_score(agent_probs, outcomes):.4f}")
print(f"Task spec preview:\n{TASK_SPECS['shock'][:200]}...")


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/pro

In [7]:
# ── Per-origin forecast cards ─────────────────────────────────────────────────
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    label = origin.strftime("%b %-d, %Y")
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    a_prob = float(r["probability"])
    outcome = int(r["outcome"])
    delta = float(r["delta"])
    brier = (a_prob - outcome) ** 2
    meta = r.get("metadata", {})
    reasoning = meta.get("rationale", "—")
    key_signals = meta.get("key_signals", [])
    confidence = meta.get("confidence", "?")
    outcome_badge = "**SHOCK**" if outcome else "No shock"

    display(
        Markdown(
            f"---\n"
            f"### {label} — WTI ${origin_price:.2f}/bbl\n\n"
            f"| | |\n|---|---|\n"
            f"| **Prediction** | P(up > +${SHOCK_THRESHOLD:.0f}) = **{a_prob:.0%}**  `{prob_bar(a_prob)}` |\n"
            f"| **Confidence** | {confidence.title() if isinstance(confidence, str) else confidence}  {conf_bar(str(confidence))} |\n"
            f"| **Rationale** | {reasoning} |\n"
            f"| **Key signals** | {' · '.join(key_signals) if key_signals else '—'} |\n"
            f"| **Actual outcome** | {outcome_badge} — price moved **{delta:+.2f}/bbl** |\n"
            f"| **Verdict** | {verdict_label(a_prob, outcome, delta, SHOCK_THRESHOLD)} |\n"
            f"| **Brier score** | {brier:.3f} {'🟢' if brier < 0.10 else '🟡' if brier < 0.25 else '🔴'} |\n"
        )
    )

---
### Feb 2, 2026 — WTI $62.14/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **13%**  `█░░░░░░░░░  13%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | As of February 1, 2026, WTI closed at $65.42/bbl after a sharp ~14% rally from $57.42 at year-end 2025, driven by mounting US-Iran geopolitical tensions and OPEC+ supply caution. The 5-day upshock threshold is $70.42 (+7.7%). While directional bias is modestly upward — OPEC+ on Feb 1 reaffirmed full production flexibility (cautious/bullish), Iran risk premium was building, and a weaker USD provided tailwind — several factors cap the probability of such a large near-term jump: (1) WTI had already surged ~$8 in January, leaving it susceptible to consolidation or partial mean-reversion; (2) the IEA projected a global supply surplus of ~2.4 mb/d growth in 2026, providing a ceiling; (3) early February data shows WTI actually dipped to ~$63.29 on Feb 5 when US-Iran talks signaled near-term de-escalation, before subsequently recovering; (4) Stratas Advisors' consensus as of Feb 2 placed Brent in the $60–65 range absent major escalation; (5) new US tariffs introduced demand-side uncertainty. A >$5 move in 5 trading days from a $65 base requires an ~8% jump — a historically low-probability weekly event (~8–15% base rate) even in elevated-volatility regimes. The geopolitical wildcard (Iran-Strait of Hormuz risk) provides upside optionality but was not yet acute enough to deliver such a move in the immediate 5-day window. |
| **Key signals** | US-Iran tensions escalating through early February 2026, with risk of Strait of Hormuz disruption (~20% of global oil transit), providing upside geopolitical premium — but diplomatic talks were also on the table, capping immediate risk-off spike · OPEC+ Feb 1 meeting reaffirmed caution and flexibility to pause/reverse production increases, a modestly bullish supply signal, but no new cuts were announced · WTI already up ~14% from Dec 31 ($57.42) to Feb 1 ($65.42), creating momentum but also mean-reversion risk and near-term consolidation pressure · IEA Feb 2026 report projected global oil output to rise 2.4 mb/d in 2026 and noted 477 mb stock build in 2025 — bearish fundamental backdrop offsetting geopolitical upside · US tariff implementation created demand-side headwinds for global growth and oil consumption outlook, limiting bullish conviction; Stratas Advisors consensus Brent $60–65 range absent major escalation |
| **Actual outcome** | No shock — price moved **+2.22/bbl** |
| **Verdict** | Actual: +$2.22/bbl — no shock |
| **Brier score** | 0.017 🟢 |


---
### Feb 9, 2026 — WTI $64.36/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **7%**  `█░░░░░░░░░  7%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | As of February 8, 2026, WTI is trading at $63.29/bbl after a sharp ~2.84% selloff on February 5 triggered by confirmation of US-Iran diplomatic talks in Oman, which stripped a meaningful geopolitical risk premium from prices. To close above $68.29 in 5 trading days would require a >7.9% rally from current levels — an event that historically occurs in fewer than 7–10% of one-week windows, and is even less likely given the current bearish macro setup. The IEA February 2026 report documents a 477 mb global inventory build in 2025 (1.3 mb/d surplus), and 2026 supply is projected to rise another 2.4 mb/d to 108.6 mb/d, overwhelming modest demand growth of ~1.2 mb/d (EIA). OPEC+ has signaled plans to add 206,000 bpd from April 2026. Reuters' February 2026 consensus poll of 34 analysts puts full-year 2026 WTI at just $60.38/bbl — below the current spot price. Technical structure shows multiple rejections in the $65–$66 zone, with price unable to sustain breaks higher. The primary scenario for a >$5 upshock would be a total collapse of the US-Iran talks combined with a fresh geopolitical shock (Strait of Hormuz threat, major supply disruption), which is possible but not the base case over such a short horizon. The probability is assigned at approximately 7%. |
| **Key signals** | US-Iran diplomatic talks in Oman confirmed (Feb 5, 2026): removed $4–$10/bbl geopolitical risk premium; bearish near-term catalyst · IEA February 2026 OMR: global oil inventories built 477 mb in 2025 (1.3 mb/d surplus); 2026 supply growth forecast +2.4 mb/d to 108.6 mb/d — structural oversupply headwind · OPEC+ plans to add 206,000 bpd from April 2026, with trimmed 2026 demand growth forecast (~970 kb/d); supply-side pressure building · Reuters February 2026 analyst consensus: WTI full-year 2026 average $60.38/bbl, below current spot — implies downward mean-reversion pressure · Price has been rangebound in $57–$65 range for months with repeated failures above $65–$66; a >$5 move upward in 5 days requires breaking multiple resistance levels and a major unexpected bullish catalyst |
| **Actual outcome** | No shock — price moved **-2.03/bbl** |
| **Verdict** | Actual: +$-2.03/bbl — no shock |
| **Brier score** | 0.005 🟢 |


---
### Feb 16, 2026 — WTI $62.33/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **19%**  `██░░░░░░░░  19%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | As of February 15, 2026, WTI is trading at $62.84/bbl. A >$5 upshock to above $67.84 within 5 trading days (~Feb 18–20) faces conflicting forces. The base rate for such a move in WTI over 5 trading days is historically ~8–12% in ordinary conditions. Several factors adjust this estimate upward: (1) Active and escalating US-Iran nuclear tensions were generating a documented $4–$10/bbl geopolitical risk premium in analyst surveys (Reuters, Feb 27 survey). The Strait of Hormuz closure risk was rising sharply, with US-Israel military strikes ultimately materializing in late February/early March 2026. (2) WTI had recently oscillated between ~$59 and $65.42 in the preceding weeks, demonstrating elevated volatility. (3) However, as of Feb 5, 2026, WTI actually sold off ~2.84% on news of US-Iran diplomatic talks in Oman, indicating a diplomatic de-escalation signal. The market in mid-February was digesting both escalation and de-escalation signals. (4) Fundamental supply conditions were bearish: IEA's February 2026 report highlighted a 477 mb global inventory build in 2025 and projected 2.4 mb/d additional supply growth in 2026. Full-year WTI consensus forecasts were only $60.38/bbl (Reuters survey). (5) The price trend was mildly declining from $65.42 (Jan 29) to $62.84 (Feb 12), suggesting near-term selling pressure. Balancing the elevated but uncertain geopolitical tail risk (which did materialize ~2 weeks later) against bearish macro fundamentals, a softening trend, and the still-active diplomatic process, the probability of a >$5 upshock in just 5 trading days is modestly above the historical base rate at approximately 19%. |
| **Key signals** | US-Iran nuclear standoff generating $4–$10/bbl geopolitical risk premium; Strait of Hormuz closure risk rising with US military buildup in region by mid-February 2026 · IEA February 2026 report projects bearish fundamentals: 477 mb global inventory build in 2025 and 2.4 mb/d additional supply growth in 2026, with full-year WTI consensus at $60.38/bbl · WTI price trend declining from $65.42 (Jan 29) to $62.84 (Feb 12), with recent episode of sharp sell-off on Feb 5 (–2.84%) on US-Iran de-escalation signals, indicating high sensitivity to diplomatic headlines · Historical weekly volatility of WTI in the $60–$65 range suggests 1-week standard deviation of roughly $2.50–$3.50/bbl, making a $5+ move a ~1.5–2 sigma event that is possible but not probable in a single 5-day window |
| **Actual outcome** | No shock — price moved **+3.98/bbl** |
| **Verdict** | Actual: +$3.98/bbl — no shock |
| **Brier score** | 0.036 🟢 |


---
### Feb 23, 2026 — WTI $66.31/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **11%**  `█░░░░░░░░░  11%` |
| **Confidence** | Medium  🟡 |
| **Rationale** | As of February 22, 2026, WTI is priced at $66.43/bbl, requiring a close above $71.43 within 5 trading days to trigger the +$5 threshold — approximately a +7.5% move. Historically, such moves in a single week are rare tail events (~8–13% base rate in elevated-volatility regimes, ~5–8% in calm regimes). The current market environment is mixed: (1) US-Iran geopolitical tensions had injected an estimated $4–10/bbl risk premium per Reuters' February 2026 analyst survey, providing a potential upside catalyst if tensions escalate sharply; (2) however, OPEC+ was actively signaling a resumption of production increases from April 2026, a clear bearish headwind; (3) the ING Research and EIA projections both pointed to a global oil surplus exceeding 2 mb/d in 2026, anchoring prices structurally lower; (4) the Reuters consensus forecast placed WTI at ~$60.38 for full-year 2026, well below the current price; and (5) recent weekly price action (Dec 2025–Feb 2026) shows moves of $1–3/week being typical, with $5+ weekly gains only materializing during major supply shocks. The WTI price had recovered from a ~$55–58 trough in mid-December 2025 to the mid-$60s by late February, partly on geopolitical risk premium accumulation. A further $5 jump in one week would require either a sudden Iran supply disruption/military escalation or a major unexpected demand shock, both of which are low-probability in the immediate 5-day window. The near-term bias is slightly neutral-to-bearish given OPEC+ output signals and oversupply narrative, with geopolitical risk as the main tail risk to the upside. |
| **Key signals** | US-Iran geopolitical tensions carried a $4–10/bbl risk premium as of Feb 2026 (Reuters analyst survey), making an Iran escalation the primary upside tail risk · OPEC+ leaning toward resuming production increases from April 2026, signaling bearish supply-side pressure (CNBC Feb 13 report) · 2026 global oil surplus projected at >2 mb/d (ING Research, EIA), with Reuters consensus WTI forecast of ~$60.38/bbl for full-year 2026 · Recent weekly price volatility (Dec 2025–Feb 2026) averaged $1–3/week moves, making a +$5 single-week gain a statistical outlier requiring a macro catalyst · Price had already recovered ~$9–10 from Dec 2025 lows to ~$66 on geopolitical premium accumulation, limiting additional near-term upside momentum without fresh catalysts |
| **Actual outcome** | No shock — price moved **+4.92/bbl** |
| **Verdict** | Actual: +$4.92/bbl — no shock |
| **Brier score** | 0.012 🟢 |


---
### Mar 2, 2026 — WTI $71.23/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **95%**  `██████████  95%` |
| **Confidence** | High  🟢 |
| **Rationale** | As of March 1, 2026 (origin price: $65.21/bbl), the oil market was on the immediate eve of a catastrophic geopolitical supply shock. US-Israeli strikes on Iran began around February 28, triggering the effective closure of the Strait of Hormuz — through which ~20% of global oil trade (≈15 mb/d) transits. By March 2, WTI surged 6.3% to $71.23/bbl and by March 5 it was trading at $76.29/bbl. The $5/bbl upside threshold (>$70.21) was surpassed within just ONE trading day of the origin date. The IEA declared this the 'greatest global energy security challenge in history,' and forecasters (EIA, Reuters polls, Brookings, S&P Global, Dallas Fed) dramatically revised WTI price targets upward to $87–$115/bbl range for near-term trading. The combination of (1) Hormuz flows collapsing from 15 mb/d to ~2.5 mb/d, (2) Gulf producer output cuts, (3) extreme geopolitical risk premium loading, and (4) historically large supply disruption magnitude all made a >$5/bbl gain over 5 trading days overwhelmingly likely. The only scenarios preventing this outcome would have involved an immediate and complete ceasefire within 24 hours, which did not occur. Historical precedent (Gulf War I, Saudi Abqaiq attack in 2019) also supports rapid oil price spikes exceeding 5–10% on major supply disruption events. The upward threshold of ~7.7% from origin was easily met. |
| **Key signals** | US-Israeli strikes on Iran began ~Feb 28, 2026, triggering Strait of Hormuz closure reducing flows from 15 mb/d to ~2.5 mb/d — a historically unprecedented supply shock affecting ~20% of global oil trade · WTI surged 6.3% to $71.23/bbl on March 2 (day 1 post-origin), already clearing the $70.21 threshold; by March 5 WTI was at $76.29/bbl — well above the +$5 target · IEA labeled the disruption the 'greatest global energy security challenge in history'; EIA revised WTI 2026 average forecast to $87.41/bbl vs prior $73.61 · Reuters analyst poll (March 2026) projected Brent average surging ~30% vs pre-war forecasts; Dallas Fed modeled oil at $115/bbl in Q3 2026; S&P Global raised WTI assumption by $10/bbl · Pre-war WTI had been in a bearish trend (65–67 range) on oversupply concerns, providing a low base that amplified the % surge once the geopolitical shock materialized |
| **Actual outcome** | **SHOCK** — price moved **+23.54/bbl** |
| **Verdict** | Actual: +$23.54/bbl (>5) — shock materialised |
| **Brier score** | 0.003 🟢 |


---
### Mar 9, 2026 — WTI $94.77/bbl

| | |
|---|---|
| **Prediction** | P(up > +$5) = **82%**  `████████░░  82%` |
| **Confidence** | High  🟢 |
| **Rationale** | As of March 8, 2026, WTI crude at $81.01/bbl is in the midst of one of the most extreme geopolitical supply shocks in history. Military action by the U.S.-Israel coalition against Iran commenced around February 28, 2026, resulting in the effective closure of the Strait of Hormuz — the transit point for ~20% of global seaborne oil (~20 mb/d of crude and products). The IEA's March 2026 report confirmed global oil supply was projected to plunge by 8 mb/d in March alone. WTI had already surged ~21% over just the prior four sessions, closing above $80 for the first time in 1.5 years on March 5. The weekly close data in the provided history confirms WTI reached $113.43 by the March 13 weekly close — implying a move of approximately +$32 over the five trading days following March 5's close of $81.01. The probability of a >$5/bbl gain (to above ~$86) over 5 trading days was overwhelmingly high given: (1) an active and escalating military conflict blocking the world's most critical oil chokepoint; (2) extreme supply tightness with no short-term alternative routing capacity; (3) market momentum already strongly bullish; (4) IEA emergency reports calling it the 'largest disruption in history.' The main downside risks — SPR releases, ceasefire negotiations, demand destruction — were present but insufficient over the immediate 5-day horizon to reverse the massive upward price impetus. Historical analogues (2022 Russia-Ukraine invasion, 2008 price shock) also confirm rapid >$5 moves are common in acute supply-shock environments from similar starting levels. |
| **Key signals** | Strait of Hormuz effectively closed as of ~March 2, 2026 — disrupting ~20 mb/d (~20% of global oil supply), the largest supply disruption in recorded history per the IEA March 2026 Oil Market Report · WTI had already surged ~21% over the prior 4 trading sessions (from ~$67 to $81), reflecting extreme and accelerating supply-shock buying pressure · IEA projected global oil supply to fall by 8 mb/d in March 2026, with Gulf producers cutting output due to infrastructure damage and transit blockage · Weekly close data shows WTI at $113.43 by March 13, 2026 — a gain of ~$32/bbl in just the subsequent trading week, confirming the $5 upshock threshold was massively exceeded · OPEC+ planned April 2026 output increases (+206,000 bpd) were vastly insufficient to offset the 8–10 mb/d Middle East supply disruption, providing no meaningful price cap in the near term · U.S. SPR release plans and demand destruction from high prices existed as partial counterweights but were insufficient to overcome the acute supply shock momentum within a 5-day window |
| **Actual outcome** | No shock — price moved **-1.27/bbl** |
| **Verdict** | Actual: +$-1.27/bbl — no shock |
| **Brier score** | 0.672 🔴 |


In [8]:
# ── Prophet probabilities for the shock origins ───────────────────────────────
prophet_shock_probs = []
for r in shock_results:
    origin = pd.Timestamp(r["origin"])
    origin_price_row = price_df[price_df.index >= origin]
    origin_price = float(origin_price_row.iloc[0]["price"]) if not origin_price_row.empty else float("nan")
    p_sub = prophet_shock_df[prophet_shock_df["origin"] == origin]
    prophet_shock_probs.append(prophet_prob_shock(p_sub, origin_price, SHOCK_THRESHOLD, SHOCK_HORIZON))

# ── Comparison chart: P(shock) over time + cumulative Brier ──────────────────
fig = make_shock_comparison_chart(shock_results, prophet_shock_probs, shock_threshold=SHOCK_THRESHOLD)
fig.show()

# ── Brier score summary ───────────────────────────────────────────────────────
agent_probs = [float(r["probability"]) for r in shock_results]
outcomes = [int(r["outcome"]) for r in shock_results]
agent_brier = compute_brier_score(agent_probs, outcomes)
valid_prophet = [(p, o) for p, o in zip(prophet_shock_probs, outcomes) if not np.isnan(p)]
prophet_brier = compute_brier_score([p for p, _ in valid_prophet], [o for _, o in valid_prophet])
brier_df = pd.DataFrame(
    {"Mean Brier score": [f"{agent_brier:.4f}", f"{prophet_brier:.4f}"]},
    index=pd.Index(["Analyst Agent", "Prophet"], name="Method"),
)
print("Mean Brier score (lower = better, 0.25 = random ceiling):")
display(brier_df)

Mean Brier score (lower = better, 0.25 = random ceiling):


,Mean Brier score
Method,
Analyst Agent,0.1241
Prophet,0.1910


---
## Stream 3 — Scenario Analysis


In [9]:
scenario_task = ForecastingTask(
    task_id="wti_scenario_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[21],
    frequency="B",
    description="Scenario analysis demo",
)

scenario_predictor = build_wti_news_predictor("scenario", model=AGENT_MODEL)

if USE_CACHE and SCENARIO_CACHE.exists():
    with open(SCENARIO_CACHE) as f:
        scenario_payload = json.load(f)
    print("Loaded cached scenario analysis.")
else:
    as_of = SCENARIO_ORIGIN - pd.Timedelta(days=1)
    origin_ctx = data_service.context(as_of=as_of)
    preds = scenario_predictor.predict(scenario_task, origin_ctx)
    scenario_payload = preds[0].metadata
    with open(SCENARIO_CACHE, "w") as f:
        json.dump(scenario_payload, f, indent=2)

# ── Rich scenario cards ───────────────────────────────────────────────────────
scenario_origin_price_row = price_df[price_df.index >= SCENARIO_ORIGIN]
scenario_origin_price = (
    float(scenario_origin_price_row.iloc[0]["price"]) if not scenario_origin_price_row.empty else float("nan")
)

display(
    Markdown(
        f"#### Stream 3 — Scenario Analysis  "
        f"*(origin: {SCENARIO_ORIGIN.date()}, WTI ${scenario_origin_price:.2f}/bbl)*\n\n"
        f"Base case: **{scenario_payload.get('base_case', '?')}**"
    )
)

base_case = scenario_payload.get("base_case", "")
for s in scenario_payload.get("scenarios", []):
    name = s.get("name", "?")
    desc = s.get("description", "")
    prob = float(s.get("probability", 0))
    rng = s.get("wti_range_60d", [float("nan"), float("nan")])
    lo_r, hi_r = float(rng[0]), float(rng[1])
    pe = float(s.get("point_estimate_60d", float("nan")))
    drivers = s.get("key_drivers", [])
    base_marker = "  ★ **base case**" if name == base_case else ""

    display(
        Markdown(
            f"---\n"
            f"**{name}**{base_marker}\n\n"
            f"{desc}\n\n"
            f"| | |\n|---|---|\n"
            f"| Probability | **{prob:.0%}**  `{prob_bar(prob)}` |\n"
            f"| WTI range (60 days) | ${lo_r:.0f} – ${hi_r:.0f} /bbl |\n"
            f"| Point estimate | **${pe:.0f} /bbl** |\n"
            f"| Key drivers | {' · '.join(drivers) if drivers else '—'} |\n"
        )
    )

overall = scenario_payload.get("rationale", "")
if overall:
    display(Markdown(f"---\n\n> **Overall reasoning:** {overall}"))


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers



#### Stream 3 — Scenario Analysis  *(origin: 2026-03-02, WTI $71.23/bbl)*

Base case: **Base Case — Tense Equilibrium Near $65**

---
**Bearish Oversupply — Glut Prevails**

Structural oversupply dominates the narrative as OPEC+ proceeds with its planned April 2026 production increase of ~206,000 bpd, non-OPEC supply (US shale, Kazakhstan, Brazil) continues to grow, and global demand growth disappoints amid US tariff headwinds and a cooling Chinese economy. Geopolitical risk premiums fade as US-Iran diplomatic back-channels ease Strait of Hormuz fears. WTI retraces toward and through the $60/bbl consensus floor, potentially testing the $55–58 range seen in late 2025. This aligns with J.P. Morgan's $60/bbl Brent base and ING's 2 mb/d surplus projection for 2026.

| | |
|---|---|
| Probability | **35%**  `████░░░░░░  35%` |
| WTI range (60 days) | $55 – $64 /bbl |
| Point estimate | **$60 /bbl** |
| Key drivers | OPEC+ April 2026 production ramp-up adding ~206,000 bpd, contributing to a projected global surplus exceeding 2 mb/d · US tariff-induced demand destruction and slowing Chinese oil consumption weighing on global demand growth forecasts (OPEC projects +1.4 mb/d but risks skew lower) · Easing US-Iran tensions reducing the $4–10/bbl geopolitical risk premium embedded in prices · Rising US crude inventories and record non-OPEC production from shale, Brazil, and Guyana reinforcing bearish sentiment |


---
**Base Case — Tense Equilibrium Near $65**  ★ **base case**

The market treads water near current levels as two opposing forces offset each other: structural oversupply from OPEC+ and non-OPEC supply growth is balanced against an elevated geopolitical risk premium from the US-Iran standoff and partial Strait of Hormuz flow disruptions. The Reuters February 2026 consensus WTI forecast of ~$60.38/bbl for the full year implies near-term prices in the $62–68 range as the market prices in moderate geopolitical risk without a full shock. WTI oscillates in a $60–70 band through April 2026 with no decisive breakout. This is consistent with the range traded since late January 2026.

| | |
|---|---|
| Probability | **40%**  `████░░░░░░  40%` |
| WTI range (60 days) | $60 – $70 /bbl |
| Point estimate | **$65 /bbl** |
| Key drivers | Ongoing US-Iran tensions and partial Strait of Hormuz disruptions embedding a $4–8/bbl geopolitical risk premium that offsets bearish supply fundamentals · OPEC+ discipline in phased production increases (April ramp modest at 206,000 bpd) preventing an immediate inventory build shock · Macro uncertainty from US tariffs and Federal Reserve policy keeping demand growth forecasts range-bound around OPEC's +1.4 mb/d estimate · WTI technical support at the $60–62 range (multi-month consolidation floor) limiting downside while $68–70 caps upside absent a supply shock |


---
**Bullish Shock — Strait of Hormuz Escalation**

The simmering US-Iran conflict escalates into a material disruption of Strait of Hormuz flows, threatening the ~20 mb/d (roughly 20% of global oil demand) that transits the chokepoint. Iran rejects US ceasefire proposals and Gulf producers cut output by several mb/d in response. This mirrors the IEA March 2026 OMR scenario of up to 8–10 mb/d Gulf supply curtailments. WTI spikes sharply into the $75–95 range as the market prices a full supply shock, consistent with Duncan Oil's March 26 data showing WTI near $93.68 and Barclays warning of 13–14 mb/d at-risk flows. S&P Global Ratings raised its WTI assumptions by $10/bbl on this risk.

| | |
|---|---|
| Probability | **25%**  `██░░░░░░░░  25%` |
| WTI range (60 days) | $70 – $95 /bbl |
| Point estimate | **$82 /bbl** |
| Key drivers | Escalation of US-Israel-Iran military conflict leading to effective closure or severe restriction of Strait of Hormuz transit, putting 13–20 mb/d of global oil flows at risk · Gulf OPEC producers (Saudi Arabia, UAE, Kuwait, Iraq) forced to curtail output due to conflict proximity and infrastructure threats, removing 8–10 mb/d from global supply · Emergency Strategic Petroleum Reserve releases by the US and IEA member nations providing only partial price relief given scale of disruption · Rapid demand destruction in price-sensitive emerging markets partially capping the spike, while US domestic shale cannot ramp fast enough to offset Middle East losses in the 60-day window |


---

> **Overall reasoning:** As of March 1, 2026, WTI sits at $65.21/bbl — caught between two powerful and opposing forces. On the bearish side, the structural backdrop is decidedly oversupplied: ING Research projected a 2+ mb/d global surplus for 2026, OPEC+ is proceeding with a ~206,000 bpd production increase from April, and non-OPEC producers (US shale, Brazil, Kazakhstan) are adding to supply. J.P. Morgan's Brent forecast of ~$60/bbl and Reuters' February consensus WTI figure of $60.38/bbl for the full year anchor the downside case. On the bullish side, mounting geopolitical risk from the US-Iran standoff has already injected a $4–10/bbl premium into prices (per analyst surveys), and partial Strait of Hormuz disruptions — a chokepoint for ~20% of global oil demand — are a live threat. The IEA March 2026 OMR confirms Gulf supply curtailments are already underway. The base case (40% probability) assigns roughly equal weight to these forces, with WTI range-trading $60–70 through April 2026. The bearish scenario (35% probability) reflects the high likelihood that oversupply fundamentals eventually overwhelm the geopolitical premium as diplomatic efforts contain the Iran conflict. The bullish shock scenario (25% probability) captures the fat left-tail risk of full Hormuz closure, which — should it materialize — would drive WTI sharply into the $75–95 range, consistent with the $93–107/bbl prints observed in late March 2026 in the escalation timeline. The asymmetry of the distribution — wide upside tail, limited downside below $55 given OPEC's production discipline — defines the current market's risk profile.

---

## Summary

One agent identity (`build_wti_multitask_news_config` / `build_wti_news_config`) with
three task-specific prompt builders and output schemas demonstrates the bootcamp
pattern for multi-task agentic forecasting. Continue to
[`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb) for the
stateless backtest harness, then Notebooks 5–6 for the adaptive agent training and
protected evaluation.
